<a href="https://colab.research.google.com/github/rahavi-r31/ExporterAI_Chapter_68_analytics/blob/colab/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# data preparation

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
data1=pd.read_excel('/content/clean HS 6.xlsx')

In [ ]:
data1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73130 entries, 0 to 73129
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   SB DATE              73130 non-null  datetime64[ns]
 1   FOB INR              73130 non-null  float64       
 2   QUANTITY             73130 non-null  float64       
 3   QUANTITY UNIT        73130 non-null  object        
 4   IMPORTER             73130 non-null  object        
 5   EXPORTER             73130 non-null  object        
 6   HS CODE              73130 non-null  int64         
 7   PRODUCT DESCRIPTION  73130 non-null  object        
 8   FOREIGN COUNTRY      73130 non-null  object        
 9   FOREIGN PORT         73130 non-null  object        
 10  INDIAN PORT          73130 non-null  object        
 11  CHAPTER              73130 non-null  int64         
 12  Year                 73130 non-null  int64         
 13  Month                73130 non-

In [ ]:
# Selection of specific columns
data = data1[['SB DATE', 'FOB INR', 'QUANTITY', 'QUANTITY UNIT' , 'IMPORTER', 'EXPORTER', 'HS CODE', 'PRODUCT DESCRIPTION', 'FOREIGN COUNTRY', 'FOREIGN PORT', 'INDIAN PORT', 'CHAPTER']]

In [ ]:
# Re-run parsing
data["SB DATE_parsed"] = pd.to_datetime(data["SB DATE"], errors="coerce", infer_datetime_format=True)

# Find rows that became NaT
failed_rows = data[data["SB DATE_parsed"].isna()]["SB DATE"].unique()
print(failed_rows)


In [ ]:
from datetime import datetime

def parse_any_date(x):
    # 1) Fast exits
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime)):
        return pd.to_datetime(x)


    if isinstance(x, (int, float)) and np.isfinite(x):
                try:
            return pd.to_datetime(x, unit="d", origin="1899-12-30")
        except Exception:
            pass


    s = str(x).strip().replace("\u00a0", " ")

    for fmt in ("%d-%m-%Y", "%d-%m-%y", "%b %d, %Y"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue

     return pd.to_datetime(s, errors="coerce", dayfirst=True)

data["SB DATE"] = data["SB DATE"].apply(parse_any_date)

unparsed = data["SB DATE"].isna()
if unparsed.any():
    print("Unparsed examples:")
    print(data.loc[unparsed, "SB DATE"].head(10))

data.drop(columns=["SB DATE_parsed"], inplace=True)

In [ ]:

data["FOB INR"] = (
    data["FOB INR"]
    .astype(str)
    .str.strip()
    .str.replace(",", "", regex=False)
    .replace("UNKNOWN", None)
    .astype(float)
)
data["QUANTITY"] = (
    data["QUANTITY"]
    .astype(str)
    .str.strip()
    .str.replace(",", "", regex=False)
    .replace("UNKNOWN", None)
    .astype(float)
)

In [ ]:
data[["QUANTITY", "FOB INR"]] = data[["QUANTITY", "FOB INR"]].fillna(0)

In [ ]:
# List of values that are not acceptable
invalid_values = ["#REF!", "#NAME?", "nana", "NA", "#N/A", "Na", "N A","N?A","Null","NOT FOUND"]

# Replace in the whole DataFrame
data = data.replace(invalid_values, "UNKNOWN")
data = data.fillna("UNKNOWN")

In [ ]:
unknown_counts = (data == "UNKNOWN").sum()
print(unknown_counts)

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
'''data[data["IEC"].isnull()]
mask = data["IEC"].isna() & data1["Item_Category_Description"].notna() #checks if iec value is present in the particular column and makes a list of true or false
data.loc[mask, "IEC"] = data1.loc[mask, "Item_Category_Description"] #copy the value to iec'''

In [ ]:
data.describe()

In [ ]:
data.info()

# Product Description

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Step 1: Create mapping for categories (from official HSN descriptions)
data["HS CODE"] = data["HS CODE"].astype(str).str.zfill(8)
data["Chapter"] = data["HS CODE"].str[:2]
data["Heading"] = data["HS CODE"].str[:4]
data["Sub Heading"] = data["HS CODE"].str[:6]
data["Tariff Item"] = data["HS CODE"]

chapter_mapping = {
    "52": "Cotton"
}

heading_mapping = {
    "5208": "Woven fabrics of cotton"
}

subheading_mapping = {
    "520811": "Unbleached, plain weave ≤100 g/m²",
    "520812": "Bleached, plain weave >100 g/m²",
    "520813": "Dyed, plain weave",
    "520819": "Other woven cotton fabrics"
}

tariff_mapping = {
    "52081130": "Shirting fabrics",
    "52081140": "Casement",
    "52081190": "Other",
    "52081210": "Dhoti",
    "52081220": "Saree",
    "52081230": "Shirting fabrics",
    "52081240": "Casement",
    "52081250": "Sheeting",
    "52081260": "Voils",
    "52081290": "Other"
}

hsn_mapping = {
    "52081130": "Woven fabrics of cotton, unbleached, plain weave",
    "52081230": "Woven fabrics of cotton, bleached, plain weave",
    "52081310": "Woven fabrics of cotton, dyed, plain weave",
    "52081320": "Woven fabrics of cotton, yarn dyed, plain weave",
    "52081390": "Woven fabrics of cotton, other dyed, plain weave",
    "52081290": "Woven fabrics of cotton, other bleached, plain weave"
}

# Map categories (HS CODE is already int64, so we can map directly)
data["Chapter Desc"] = data["Chapter"].map(chapter_mapping)
data["Heading Desc"] = data["Heading"].map(heading_mapping)
data["Sub Heading Desc"] = data["Sub Heading"].map(subheading_mapping)
data["Tariff Desc"] = data["Tariff Item"].map(tariff_mapping).fillna("Other Cotton Fabrics")
data["CATEGORY"] = data["HS CODE"].map(hsn_mapping)

# For HS codes not in our mapping, create a general category


# Step 2: Extract sub-category clusters (optional ML step)
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000, min_df=2, max_df=0.8)
# Fixed column name typo
X = vectorizer.fit_transform(data["PRODUCT DESCRIPITION"])

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
data["SUB CATEGORY CLUSTER"] = kmeans.fit_predict(X)

# Step 3: Enhanced Multi-label sub-categorisation
def name_cluster_multi(text):
    text = text.lower()
    tags = []

    # Primary fabric usage categories
    if "shirting" in text or "shiring" in text or "shiriting" in text:
        tags.append("Shirting")
    if "suiting" in text:
        tags.append("Suiting")
    if "blazer" in text:
        tags.append("Blazer Cloth")
    if "blouse" in text or "blowas" in text:
        tags.append("Blouse Cloth")

    # Processing/finishing methods
    if "grey" in text or "greige" in text or "gray" in text:
        tags.append("Grey/Greige")
    if "yarn dyed" in text or "yarn-dyed" in text or "y/d" in text:
        if "indigo" in text:
            tags.append("Indigo Yarn Dyed")
        else:
            tags.append("Yarn Dyed")
    if "printed" in text or "print" in text:
        if "digital" in text:
            tags.append("Digital Printed")
        else:
            tags.append("Printed")
    if "dyed" in text and "yarn" not in text:
        tags.append("Dyed")
    if "bleached" in text or "white" in text:
        tags.append("Bleached/White")
    if "embroidered" in text:
        tags.append("Embroidered")
    if "mercerised" in text:
        tags.append("Mercerised")
    if "finished" in text:
        tags.append("Finished")
    if "washed" in text:
        tags.append("Washed")
    if "singed" in text:
        tags.append("Singed")

    # Premium/luxury fabric types
    if any(word in text for word in ["silk", "chiffon", "georgette", "crepe", "satin", "velvet"]):
        tags.append("Premium/Luxury")
    if "crape" in text:
        tags.append("Premium/Luxury")

    # Traditional Indian fabrics
    if any(word in text for word in ["khadi", "handloom", "chanderi", "kora", "jamdani", "ikat", "pattu"]):
        tags.append("Traditional Indian")
    if "chikan" in text or "chicken" in text:
        tags.append("Chikan Embroidery")
    if "malmal" in text or "malaml" in text:
        tags.append("Malmal (Muslin)")
    if "markin" in text or "markeen" in text or "markeen" in text:
        tags.append("Markin Cotton")
    if "rubiya" in text or "rubbya" in text:
        tags.append("Rubiya Cotton")
    if "dashna" in text or "dasna" in text:
        tags.append("Dashna Cotton")
    if "patra" in text:
        tags.append("Patra Cotton")
    if "chalte" in text or "chaltan" in text or "chalteen" in text:
        tags.append("Chalti/Chalte Cotton")
    if "tanna" in text:
        tags.append("Tanna Cotton")

    # Specialty cotton weaves and constructions
    if "poplin" in text or "papleen" in text or "papilen" in text:
        tags.append("Poplin")
    if "gadda" in text:
        tags.append("Mattress Cloth")
    if any(word in text for word in ["cambric", "lawn", "muslin", "voile", "percale", "sateen"]):
        tags.append("Fine Cotton Weaves")
    if any(word in text for word in ["twill", "dobby", "jacquard", "herringbone", "waffle"]):
        tags.append("Structured Weaves")
    if "flannel" in text:
        tags.append("Flannel")
    if any(word in text for word in ["dhaka", "mulmul", "malmal", "rubia"]):
        tags.append("Traditional Cotton Varieties")

    # Knit fabrics
    if any(word in text for word in ["jersey", "rib", "interlock", "single jersey", "double jersey", "stokinett"]):
        tags.append("Knit/Jersey")

    # Heavy duty/canvas
    if any(word in text for word in ["canvas", "duck", "heavy", "workwear", "sofa", "upholstery"]):
        tags.append("Heavy Duty")
    if "denim" in text or "jeans" in text:
        tags.append("Denim")

    # Synthetic and man-made fibers
    if any(word in text for word in ["polyester", "acrylic", "nylon", "synthetic", "man made", "terricot"]) and "cotton" not in text:
        tags.append("Synthetic")

    # Eco-friendly/sustainable
    if any(word in text for word in ["organic", "recycled", "sustainable", "eco", "tencel", "lyocell", "modal", "lenzing"]):
        tags.append("Eco-Friendly")

    # Fiber content categories
    if "organic" in text:
        tags.append("Organic")
    if "combed" in text:
        tags.append("Combed")
    if ("100%" in text and "cotton" in text) or "100 percent cotton" in text or "100 cotton" in text:
        tags.append("100% Cotton")

    # Linen categories
    if "linen" in text:
        if "cotton" in text:
            tags.append("Linen-Cotton Blend")
        else:
            tags.append("Pure Linen")

    # Rayon/Viscose
    if any(word in text for word in ["rayon", "viscose"]):
        if "cotton" in text:
            tags.append("Rayon-Cotton Blend")
        else:
            tags.append("Rayon/Viscose")

    # Cotton blends (general)
    if "cotton" in text and any(word in text for word in ["polyester", "viscose", "spandex", "elastane", "blend"]):
        tags.append("Cotton Blends")

    # Home textiles
    if any(word in text for word in ["sheeting", "bedding", "towel", "table cloth", "curtain", "casement", "bedsheet", "pillow cover", "duster","razai", "quilt", "sofa"]):
        tags.append("Home Textiles")

    # Garment categories
    if any(word in text for word in ["ladies", "salwar", "saree", "lehenga", "kurta", "blouse", "dress material", "kurtee", "dupatta"]):
        tags.append("Ladies Garments")
    if any(word in text for word in ["shirt"]):
        tags.append("Shirts")
    if any(word in text for word in ["innerwear", "undergarment", "brief", "vest"]):
        tags.append("Innerwear")
    if any(word in text for word in ["uniform", "workwear", "industrial"]):
        tags.append("Uniform/Workwear")
    if any(word in text for word in ["baby", "infant", "kids", "children"]):
        tags.append("Kids/Baby")
    if any(word in text for word in ["medical", "surgical", "mask", "hospital"]):
        tags.append("Medical/Healthcare")

    # Wool fabrics
    if any(word in text for word in ["wool", "woolen", "worsted"]):
        tags.append("Wool")

    # Net/mesh
    if any(word in text for word in ["net", "mesh", "tulle"]):
        tags.append("Net/Mesh")

    # Production method tags
    if "handloom" in text:
        tags.append("Handloom")
    if "powerloom" in text:
        tags.append("Powerloom")

    if "lungi" in text:
        tags.append("Lungi (Traditional Garment)")
    if "lace" in text:
        tags.append("Lace")
    if "dhoti" in text:
        tags.append("Dhoti (Traditional Garment)")
    if "gamcha" in text:
        tags.append("Gamcha (Towel/Workwear)")
    if "jhola" in text:
        tags.append("Jhola (Bag)")
    if "razai" in text or "quilt" in text:
        tags.append("Bedding (Razai/Quilt Cover)")
    if "pooja" in text or "ceremonial" in text or "religious" in text or "puja" in text:
        tags.append("Religious/Ceremonial Cloth")
    if "free sample" in text:
        tags.append("SAMPLE")
    if any(word in text for word in ["coller", "cuff", "belt", "roll", "hook", "astar"]):
        tags.append("Garment Accessories")
    if any(word in text for word in ["anyaresha", "brand", "detail as per invoice", "gracy", "sangwa", "sanghvi", "queen", "fariya", "falatine", "california", "tesla", "invoice", "kamani", "kamni", "suplier", "supplier", "tax inv no", "bale no"]):
        tags.append("Unclassified (Brand/Invoice Reference)")
    if any(word in text for word in ["WE INTEND TO CLAIM", "MEIS"]):
        tags.append("MEIS")
    if any(word in text for word in ["WE UNDERTAKE TO ABIDE BY THE"]):
        tags.append("FIEMA")


    if "suit" in text:
        tags.append("Suits")
    if "unstich" in text or "unstitch" in text:
        tags.append("Unstitched Material")
    if "dress" in text:
        tags.append("Dress Material")
    if any(word in text for word in ["blazer", "jacket", "coat", "hoody"]):
        tags.append("Blazers / Jackets")
    if any(word in text for word in ["pant", "pents", "paint", "trouser"]):
        tags.append("Trousers / Pants")
    if "saree" in text or "sadi" in text:
        tags.append("Sarees / Ethnic Wear")
    if "bed sheet" in text or "chaddar" in text:
        tags.append("Home Furnishing")
    if any(word in text for word in ["markin", "casment", "ctn", "cottan"]):
        tags.append("Cotton Fabric")
    if any(word in text for word in ["polysoft", "polyster", "polyester"]):
        tags.append("Polyester Fabric")
    if any(word in text for word in ["fabrio", "fabrios", "fabric", "fabriana"]):
        tags.append("Fabric / Textile")
    if any(word in text for word in ["check", "checks", "cheack"]):
        tags.append("Checked Fabric")
    if "thread" in text or "yarn" in text:
        tags.append("Thread / Yarn")
    if any(word in text for word in ["garments", "readymade", "apparel"]):
        tags.append("Readymade Garments")
    if any(word in text for word in ["shawl", "rumala", "thundu"]):
        tags.append("Shawls / Religious")
    if any(word in text for word in ["assort", "mix lot", "mix colour", "assorted"]):
        tags.append("Mixed / Assorted")
    if any(word in text for word in ["export", "meis", "fema", "gst", "invoice", "epcg", "supply bill", "arn no"]):
        tags.append("Documentation / Compliance")


    # Fallback categories
    if not tags and "cotton" in text:
        tags.append("Cotton - General")
    if not tags and any(word in text for word in ["fabric", "textile", "cloth"]):
        tags.append("Other Fabrics")
    if not tags:
        tags.append("Unclassified")

    return " | ".join(tags)  # Using | separator for better readability

# Apply multi-label tagging (fixed column name)
data["SUB CATEGORY"] = data["PRODUCT DESCRIPITION"].apply(name_cluster_multi)

# Step 4: Enhanced analysis
print("=== MULTI-LABEL FABRIC ANALYSIS ===\n")

# Count individual tags
all_tags = []
for tags_string in data["SUB CATEGORY"]:
    all_tags.extend(tags_string.split(" | "))

tag_counts = pd.Series(all_tags).value_counts()
print("Top 20 Individual Tags:")
print(tag_counts.head(20))
print("\n" + "="*50 + "\n")

# Show most common tag combinations
print("Top 15 Tag Combinations:")
print(data["SUB CATEGORY"].value_counts().head(15))
print("\n" + "="*50 + "\n")

# Analyze unclassified products
unclassified = data[data["SUB CATEGORY"] == "Unclassified"]
print(f"Unclassified Products: {len(unclassified)} out of {len(data)} ({len(unclassified)/len(data)*100:.2f}%)")
if len(unclassified) > 0:
    print("\nSample Unclassified Products:")
    print(unclassified["PRODUCT DESCRIPITION"].head(10).tolist())
print("\n" + "="*50 + "\n")

# Show products with multiple tags (complex products)
multi_tag_products = data[data["SUB CATEGORY"].str.contains(r"\|")]
print(f"Products with Multiple Tags: {len(multi_tag_products)} out of {len(data)} ({len(multi_tag_products)/len(data)*100:.2f}%)")
print("\nSample Multi-Tag Products:")
sample_multi = multi_tag_products[["PRODUCT DESCRIPITION", "SUB CATEGORY"]].head(10)
for idx, row in sample_multi.iterrows():
    print(f"• {row['PRODUCT DESCRIPITION']}")
    print(f"  Tags: {row['SUB CATEGORY']}")
    print()

print("="*50)
print("Analysis Complete!")

In [ ]:
import pandas as pd
import re

# Step 1: Create mapping for categories (from official HSN descriptions)
data["HS CODE"] = data["HS CODE"].astype(str).str.zfill(8)
data["Chapter"] = data["HS CODE"].str[:2]
data["Heading"] = data["HS CODE"].str[:4]
data["Sub Heading"] = data["HS CODE"].str[:6]
data["Tariff Item"] = data["HS CODE"]

chapter_mapping = {
    "52": "Cotton"
}

heading_mapping = {
    "5208": "Woven fabrics of cotton"
}

subheading_mapping = {
    "520811": "Unbleached, plain weave ≤100 g/m²",
    "520812": "Bleached, plain weave >100 g/m²",
    "520813": "Dyed, plain weave",
    "520819": "Other woven cotton fabrics"
}

tariff_mapping = {
    "52081130": "Shirting fabrics",
    "52081140": "Casement",
    "52081190": "Other",
    "52081210": "Dhoti",
    "52081220": "Saree",
    "52081230": "Shirting fabrics",
    "52081240": "Casement",
    "52081250": "Sheeting",
    "52081260": "Voils",
    "52081290": "Other"
}

hsn_mapping = {
    "52081130": "Woven fabrics of cotton, unbleached, plain weave",
    "52081230": "Woven fabrics of cotton, bleached, plain weave",
    "52081310": "Woven fabrics of cotton, dyed, plain weave",
    "52081320": "Woven fabrics of cotton, yarn dyed, plain weave",
    "52081390": "Woven fabrics of cotton, other dyed, plain weave",
    "52081290": "Woven fabrics of cotton, other bleached, plain weave"
}

# Map categories (HS CODE is already int64, so we can map directly)
data["Chapter Desc"] = data["Chapter"].map(chapter_mapping)
data["Heading Desc"] = data["Heading"].map(heading_mapping)
data["Sub Heading Desc"] = data["Sub Heading"].map(subheading_mapping)
data["Tariff Desc"] = data["Tariff Item"].map(tariff_mapping).fillna("Other Cotton Fabrics")
data["CATEGORY"] = data["HS CODE"].map(hsn_mapping)

# Step 2: Keyword-based tagging only
def name_cluster_multi(text):
    text = str(text).lower()
    tags = []

    # Primary fabric usage categories
    if "shirting" in text or "shiring" in text or "shiriting" in text:
        tags.append("Shirting")
    if "suiting" in text:
        tags.append("Suiting")
    if "blazer" in text:
        tags.append("Blazer Cloth")
    if "blouse" in text or "blowas" in text:
        tags.append("Blouse Cloth")

    # Processing/finishing methods
    if "grey" in text or "greige" in text or "gray" in text:
        tags.append("Grey/Greige")
    if "yarn dyed" in text or "yarn-dyed" in text or "y/d" in text:
        if "indigo" in text:
            tags.append("Indigo Yarn Dyed")
        else:
            tags.append("Yarn Dyed")
    if "printed" in text or "print" in text:
        if "digital" in text:
            tags.append("Digital Printed")
        else:
            tags.append("Printed")
    if "dyed" in text and "yarn" not in text:
        tags.append("Dyed")
    if "bleached" in text or "white" in text:
        tags.append("Bleached/White")
    if "embroidered" in text:
        tags.append("Embroidered")
    if "mercerised" in text:
        tags.append("Mercerised")
    if "finished" in text:
        tags.append("Finished")
    if "washed" in text:
        tags.append("Washed")
    if "singed" in text:
        tags.append("Singed")

    # Premium/luxury fabric types
    if any(word in text for word in ["silk", "chiffon", "georgette", "crepe", "satin", "velvet", "crape"]):
        tags.append("Premium/Luxury")

    # Traditional Indian fabrics
    if any(word in text for word in ["khadi", "handloom", "chanderi", "kora", "jamdani", "ikat", "pattu"]):
        tags.append("Traditional Indian")
    if "chikan" in text or "chicken" in text:
        tags.append("Chikan Embroidery")
    if "malmal" in text or "malaml" in text:
        tags.append("Malmal (Muslin)")
    if "markin" in text or "markeen" in text:
        tags.append("Markin Cotton")
    if "rubiya" in text or "rubbya" in text:
        tags.append("Rubiya Cotton")
    if "dashna" in text or "dasna" in text:
        tags.append("Dashna Cotton")
    if "patra" in text:
        tags.append("Patra Cotton")
    if "chalte" in text or "chaltan" in text or "chalteen" in text:
        tags.append("Chalti/Chalte Cotton")
    if "tanna" in text:
        tags.append("Tanna Cotton")

    # Specialty cotton weaves and constructions
    if any(word in text for word in ["poplin", "papleen", "papilen"]):
        tags.append("Poplin")
    if "gadda" in text:
        tags.append("Mattress Cloth")
    if any(word in text for word in ["cambric", "lawn", "muslin", "voile", "percale", "sateen"]):
        tags.append("Fine Cotton Weaves")
    if any(word in text for word in ["twill", "dobby", "jacquard", "herringbone", "waffle"]):
        tags.append("Structured Weaves")
    if "flannel" in text:
        tags.append("Flannel")

    # Knit fabrics
    if any(word in text for word in ["jersey", "rib", "interlock", "single jersey", "double jersey", "stokinett"]):
        tags.append("Knit/Jersey")

    # Heavy duty/canvas
    if any(word in text for word in ["canvas", "duck", "heavy", "workwear", "sofa", "upholstery"]):
        tags.append("Heavy Duty")
    if "denim" in text or "jeans" in text:
        tags.append("Denim")

    # Synthetic and man-made fibers
    if any(word in text for word in ["polyester", "acrylic", "nylon", "synthetic", "man made", "terricot"]) and "cotton" not in text:
        tags.append("Synthetic")

    # Eco-friendly/sustainable
    if any(word in text for word in ["organic", "recycled", "sustainable", "eco", "tencel", "lyocell", "modal", "lenzing"]):
        tags.append("Eco-Friendly")

    # Fiber content categories
    if "organic" in text:
        tags.append("Organic")
    if "combed" in text:
        tags.append("Combed")
    if ("100%" in text and "cotton" in text) or "100 percent cotton" in text or "100 cotton" in text:
        tags.append("100% Cotton")

    # Linen categories
    if "linen" in text:
        if "cotton" in text:
            tags.append("Linen-Cotton Blend")
        else:
            tags.append("Pure Linen")

    # Rayon/Viscose
    if any(word in text for word in ["rayon", "viscose"]):
        if "cotton" in text:
            tags.append("Rayon-Cotton Blend")
        else:
            tags.append("Rayon/Viscose")

    # Cotton blends (general)
    if "cotton" in text and any(word in text for word in ["polyester", "viscose", "spandex", "elastane", "blend"]):
        tags.append("Cotton Blends")

    # Home textiles
    if any(word in text for word in ["sheeting", "bedding", "towel", "table cloth", "curtain", "casement", "bedsheet", "pillow cover", "duster","razai", "quilt", "sofa"]):
        tags.append("Home Textiles")

    # Garment categories
    if any(word in text for word in ["ladies", "salwar", "saree", "lehenga", "kurta", "blouse", "dress material", "kurtee", "dupatta"]):
        tags.append("Ladies Garments")
    if any(word in text for word in ["shirt"]):
        tags.append("Shirts")
    if any(word in text for word in ["innerwear", "undergarment", "brief", "vest"]):
        tags.append("Innerwear")
    if any(word in text for word in ["uniform", "workwear", "industrial"]):
        tags.append("Uniform/Workwear")
    if any(word in text for word in ["baby", "infant", "kids", "children"]):
        tags.append("Kids/Baby")
    if any(word in text for word in ["medical", "surgical", "mask", "hospital"]):
        tags.append("Medical/Healthcare")

    # Wool fabrics
    if any(word in text for word in ["wool", "woolen", "worsted"]):
        tags.append("Wool")

    # Net/mesh
    if any(word in text for word in ["net", "mesh", "tulle"]):
        tags.append("Net/Mesh")

    # Production method tags
    if "handloom" in text:
        tags.append("Handloom")
    if "powerloom" in text:
        tags.append("Powerloom")

    if "lungi" in text:
        tags.append("Lungi (Traditional Garment)")
    if "lace" in text:
        tags.append("Lace")
    if "dhoti" in text:
        tags.append("Dhoti (Traditional Garment)")
    if "gamcha" in text:
        tags.append("Gamcha (Towel/Workwear)")
    if "jhola" in text:
        tags.append("Jhola (Bag)")
    if "razai" in text or "quilt" in text:
        tags.append("Bedding (Razai/Quilt Cover)")
    if "pooja" in text or "ceremonial" in text or "religious" in text or "puja" in text:
        tags.append("Religious/Ceremonial Cloth")
    if "free sample" in text:
        tags.append("SAMPLE")

    # Fallback categories
    if not tags and "cotton" in text:
        tags.append("Cotton - General")
    if not tags and any(word in text for word in ["fabric", "textile", "cloth"]):
        tags.append("Other Fabrics")
    if not tags:
        tags.append("Unclassified")

    return " | ".join(tags)

# Apply tagging
data["SUB CATEGORY"] = data["PRODUCT DESCRIPTION"].apply(name_cluster_multi)

# Step 3: Summary analysis
print("=== MULTI-LABEL FABRIC ANALYSIS ===\n")

# Count individual tags
all_tags = []
for tags_string in data["SUB CATEGORY"]:
    all_tags.extend(tags_string.split(" | "))

tag_counts = pd.Series(all_tags).value_counts()
print("Top 20 Individual Tags:")
print(tag_counts.head(20))
print("\n" + "="*50 + "\n")

# Show most common tag combinations
print("Top 15 Tag Combinations:")
print(data["SUB CATEGORY"].value_counts().head(15))
print("\n" + "="*50 + "\n")

# Analyze unclassified products
unclassified = data[data["SUB CATEGORY"] == "Unclassified"]
print(f"Unclassified Products: {len(unclassified)} out of {len(data)} ({len(unclassified)/len(data)*100:.2f}%)")
if len(unclassified) > 0:
    print("\nSample Unclassified Products:")
    print(unclassified["PRODUCT DESCRIPTION"].head(10).tolist())
print("\n" + "="*50 + "\n")

# Products with multiple tags
multi_tag_products = data[data["SUB CATEGORY"].str.contains(r"\|")]
print(f"Products with Multiple Tags: {len(multi_tag_products)} out of {len(data)} ({len(multi_tag_products)/len(data)*100:.2f}%)")
print("\nSample Multi-Tag Products:")
sample_multi = multi_tag_products[["PRODUCT DESCRIPTION", "SUB CATEGORY"]].head(10)
for idx, row in sample_multi.iterrows():
    print(f"• {row['PRODUCT DESCRIPTION']}")
    print(f"  Tags: {row['SUB CATEGORY']}")
    print()

print("="*50)
print("Analysis Complete!")


In [ ]:
tag_groups = {
    "Usage": [
        "Shirting", "Suiting", "Blazer Cloth", "Blouse Cloth", "Uniform/Workwear",
        "Home Textiles", "Ladies Garments", "Kids/Baby", "Medical/Healthcare",
        "Shirts", "Innerwear", "Suits", "Unstitched Material", "Dress Material",
        "Blazers / Jackets", "Trousers / Pants", "Sarees / Ethnic Wear",
        "Home Furnishing", "Readymade Garments", "Shawls / Religious",
        "Lungi (Traditional Garment)", "Dhoti (Traditional Garment)", "Gamcha (Towel/Workwear)",
        "Jhola (Bag)", "Bedding (Razai/Quilt Cover)", "Religious/Ceremonial Cloth"
    ],
    "Processing": [
        "Grey/Greige", "Yarn Dyed", "Indigo Yarn Dyed", "Printed", "Digital Printed",
        "Dyed", "Bleached/White", "Embroidered", "Mercerised", "Finished",
        "Washed", "Singed", "Handloom", "Powerloom"
    ],
    "Material": [
        "100% Cotton", "Cotton Blends", "Cotton Fabric", "Polyester Fabric", "Fabric / Textile",
        "Premium/Luxury", "Synthetic", "Eco-Friendly", "Organic", "Combed",
        "Pure Linen", "Linen-Cotton Blend", "Rayon/Viscose", "Rayon-Cotton Blend",
        "Wool", "Denim"
    ],
    "Weave/Construction": [
        "Poplin", "Mattress Cloth", "Fine Cotton Weaves", "Structured Weaves", "Flannel",
        "Traditional Indian", "Chikan Embroidery", "Malmal (Muslin)", "Markin Cotton",
        "Rubiya Cotton", "Dashna Cotton", "Patra Cotton", "Chalti/Chalte Cotton",
        "Tanna Cotton", "Traditional Cotton Varieties", "Knit/Jersey", "Heavy Duty",
        "Net/Mesh", "Checked Fabric", "Thread / Yarn"
    ],
    "Other": [
        "SAMPLE", "Garment Accessories", "Unclassified", "Unclassified (Brand/Invoice Reference)",
        "MEIS", "FIEMA", "Mixed / Assorted", "Documentation / Compliance"
    ]
}
def split_tags_by_group(tags_string):
    tags = tags_string.split(" | ")
    result = {group: [] for group in tag_groups}  # initialise all groups

    for tag in tags:
        for group, group_tags in tag_groups.items():
            if tag in group_tags:
                result[group].append(tag)
                break
    return result

# Apply grouping
grouped_tags = data["SUB CATEGORY"].apply(split_tags_by_group)

# Flatten into separate columns
for group in tag_groups.keys():
    data[group] = grouped_tags.apply(lambda x: " | ".join(x[group]) if x[group] else None)


In [ ]:
# Filter only HS code 52081290
data = data[data["HS CODE"].astype(str).str.zfill(8) == "52081290"].copy()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

hsn_sac = pd.read_excel("/content/HSN_SAC.xlsx")  # Columns: HSN_CD, HSN_Description, Unit, Rate of Duty Standard, Rate of Duty Preferential Areas

data["HS CODE"] = data["HS CODE"].astype(str).str.zfill(8)
data["Chapter"] = data["HS CODE"].str[:2]
data["Heading"] = data["HS CODE"].str[:4]
data["Sub Heading"] = data["HS CODE"].str[:6]
data["Tariff Item"] = data["HS CODE"]

chapter_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()
heading_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()
subheading_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()
tariff_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()

data["Chapter Desc"] = data["Chapter"].map(chapter_map)
data["Heading Desc"] = data["Heading"].map(heading_map)
data["Sub Heading Desc"] = data["Sub Heading"].map(subheading_map)
data["Tariff Desc"] = data["Tariff Item"].map(tariff_map)
def build_detailed_desc(row):
    parts = []
    if pd.notna(row["Chapter Desc"]):
        parts.append(f"Chapter: {row['Chapter Desc']}")
    if pd.notna(row["Heading Desc"]):
        parts.append(f"Heading: {row['Heading Desc']}")
    if pd.notna(row["Sub Heading Desc"]):
        parts.append(f"Subheading: {row['Sub Heading Desc']}")
    if pd.notna(row["Tariff Desc"]):
        parts.append(f"Tariff: {row['Tariff Desc']}")
    return " | ".join(parts)

data["CATEGORY"] = data.apply(build_detailed_desc, axis=1)

vectorizer = TfidfVectorizer(stop_words="english", max_features=1000, min_df=2, max_df=0.8)
X = vectorizer.fit_transform(data["PRODUCT DESCRIPTION"])

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
data["SUB CATEGORY CLUSTER"] = kmeans.fit_predict(X)

def get_top_words_per_cluster(kmeans, vectorizer, n_words=5):
    """Return dictionary {cluster_id: [top words]}"""
    terms = vectorizer.get_feature_names_out()
    top_words = {}
    for i, center in enumerate(kmeans.cluster_centers_):
        top_indices = center.argsort()[-n_words:][::-1]
        top_terms = [terms[ind] for ind in top_indices]
        top_words[i] = top_terms
    return top_words

cluster_keywords = get_top_words_per_cluster(kmeans, vectorizer, n_words=5)

def name_cluster_multi(description, cluster_id, cluster_keywords, top_n=3):
    """
    Assigns a human-readable label for a cluster, based on its top keywords.
    Falls back to cluster number if no keywords.
    """
    if cluster_id in cluster_keywords:
        label = " / ".join(cluster_keywords[cluster_id][:top_n])
        return f"{label} (Cluster {cluster_id})"
    else:
        return f"Cluster {cluster_id}"

cluster_labels = {
    cid: " / ".join(words[:3]) for cid, words in cluster_keywords.items()
}

data["SUB CATEGORY"] = data.apply(
    lambda row: name_cluster_multi(
        row["PRODUCT DESCRIPTION"],
        row["SUB CATEGORY CLUSTER"],
        cluster_keywords
    ),
    axis=1
)


def explain_cluster_membership(description, cluster_id, cluster_keywords, top_n=5):
    """
    For a given row, show which words in its description overlap
    with the defining keywords of its assigned cluster.
    """
    # cluster's top words
    cluster_words = set(cluster_keywords.get(cluster_id, [])[:top_n])
    # words in description
    desc_words = set(description.lower().split())
    # overlap
    overlap = cluster_words.intersection(desc_words)

    if overlap:
        return f"Cluster {cluster_id}: {' / '.join(sorted(overlap))}"
    else:
        return f"Cluster {cluster_id}: {' / '.join(cluster_keywords.get(cluster_id, [])[:top_n])}"

# Apply for all rows
data["SUB CATEGORY (explained)"] = data.apply(
    lambda row: explain_cluster_membership(
        row["PRODUCT DESCRIPTION"],
        row["SUB CATEGORY CLUSTER"],
        cluster_keywords
    ),
    axis=1
)


review_table = data[[
    "PRODUCT DESCRIPTION","HS CODE", "Chapter", "Chapter Desc",
    "Heading", "Heading Desc",
    "Sub Heading", "Sub Heading Desc",
    "Tariff Item", "Tariff Desc",
    "CATEGORY", "SUB CATEGORY","SUB CATEGORY (explained)"
]]


print("Generalised HSN mapping + clustering completed.")'''


In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from tqdm import tqdm

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def find_optimal_clusters(X, max_clusters=15):
    """
    Use the Elbow Method to find the optimal number of clusters.
    """
    sse = []
    print("Finding optimal number of clusters...")
    for k in tqdm(range(2, max_clusters + 1)):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        sse.append(kmeans.inertia_)

    # Plot the elbow curve
    plt.figure(figsize=(8, 5))
    plt.plot(range(2, max_clusters + 1), sse, 'bx-')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Sum of Squared Distances (SSE)')
    plt.title('Elbow Method For Optimal k')
    plt.show()

    print("Review the plot to identify the 'elbow' point.")
    return None

# --- Step 1: Load and Pre-process Data ---

# Assuming 'data' and 'hsn_sac' DataFrames are already loaded
# data = pd.read_csv("transactions.csv")
hsn_sac = pd.read_excel("/content/HSN_SAC.xlsx")

# Apply meticulous data cleaning
print("Starting data cleaning...")
data["CLEANED_DESCRIPTION"] = data["PRODUCT DESCRIPTION"].apply(clean_text)

# --- Step 2: HSN-based Categorization (Your existing code) ---

data["HS CODE"] = data["HS CODE"].astype(str).str.zfill(8)
data["Chapter"] = data["HS CODE"].str[:2]
data["Heading"] = data["HS CODE"].str[:4]
data["Sub Heading"] = data["HS CODE"].str[:6]
data["Tariff Item"] = data["HS CODE"]
# --- Step 3: Map descriptions from hsn_sac ---
# Create lookup dicts for different HSN code lengths
chapter_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()
heading_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()
subheading_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()
tariff_map = hsn_sac.set_index(hsn_sac["HSN_CD"])["HSN_Description"].to_dict()

# Assign descriptions dynamically
data["Chapter Desc"] = data["Chapter"].map(chapter_map)
data["Heading Desc"] = data["Heading"].map(heading_map)
data["Sub Heading Desc"] = data["Sub Heading"].map(subheading_map)
data["Tariff Desc"] = data["Tariff Item"].map(tariff_map)
def build_detailed_desc(row):
    parts = []
    if pd.notna(row["Chapter Desc"]):
        parts.append(f"Chapter: {row['Chapter Desc']}")
    if pd.notna(row["Heading Desc"]):
        parts.append(f"Heading: {row['Heading Desc']}")
    if pd.notna(row["Sub Heading Desc"]):
        parts.append(f"Subheading: {row['Sub Heading Desc']}")
    if pd.notna(row["Tariff Desc"]):
        parts.append(f"Tariff: {row['Tariff Desc']}")
    return " | ".join(parts)

data["CATEGORY"] = data.apply(build_detailed_desc, axis=1)

# --- Step 3: Sub-category clustering on CLEANED_DESCRIPTION ---

# Use a more flexible vectorizer
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000, min_df=5, max_df=0.7)
X = vectorizer.fit_transform(data["CLEANED_DESCRIPTION"])

# Find the optimal number of clusters (manual step based on the plot)
# find_optimal_clusters(X)
# **After running this, manually select k from the plot and set n_clusters below**
# Example: Let's assume you found 12 to be the optimal number
k_clusters = 12

kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
data["SUB CATEGORY CLUSTER"] = kmeans.fit_predict(X)

# --- Step 4: Extracting and Naming Clusters (Refined) ---

def get_top_words_per_cluster(kmeans, vectorizer, n_words=7):
    """Return dictionary {cluster_id: [top words]}"""
    terms = vectorizer.get_feature_names_out()
    top_words = {}
    for i, center in enumerate(kmeans.cluster_centers_):
        top_indices = center.argsort()[-n_words:][::-1]
        top_terms = [terms[ind] for ind in top_indices]
        top_words[i] = top_terms
    return top_words

cluster_keywords = get_top_words_per_cluster(kmeans, vectorizer, n_words=7)

def name_cluster(cluster_id, cluster_keywords, top_n=3):
    """Assigns a human-readable label based on top keywords."""
    if cluster_id in cluster_keywords:
        label = " / ".join(cluster_keywords[cluster_id][:top_n])
        return label
    else:
        return f"Cluster {cluster_id}"

# Create a mapping of cluster ID to a human-readable label
cluster_labels_map = {
    cid: name_cluster(cid, cluster_keywords) for cid in range(k_clusters)
}

# Assign the SUB CATEGORY name using the map
data["SUB CATEGORY"] = data["SUB CATEGORY CLUSTER"].map(cluster_labels_map)

# --- Step 5: Review output (Your existing code) ---

review_table = data[[
    "PRODUCT DESCRIPTION", "CLEANED_DESCRIPTION", "CATEGORY",
    "SUB CATEGORY", "SUB CATEGORY CLUSTER"
]]

print("Generalised HSN mapping + clustering completed.")
print(review_table.head(10))

# Example of how to view all keywords for a specific cluster
# print(f"Keywords for Cluster 1: {cluster_keywords.get(1)}")

In [ ]:
review_table.to_excel("hsn_review_generalised.xlsx", index=False)

In [ ]:
cluster_stats = data.groupby("SUB CATEGORY CLUSTER").agg({
    "HS CODE": "nunique",          # how many unique HS codes
    "PRODUCT DESCRIPTION": "count",# total rows
    "SUB CATEGORY":"UNIQU"        # avg quantity (raw, unnormalised)
}).reset_index()
cluster_stats.columns = [
    "SUB CATEGORY CLUSTER",
    "Unique HS Codes",
    "Total Shipments",
    "SUB CATEGORY COUNT"
]

print("\n📊 Cluster-level stats:")
print(cluster_stats.head(10))

In [ ]:
data["SUB CATEGORY CLUSTER"].value_counts()

In [ ]:
# --- Display sample rows for sanity check ---
print("📌 Sample clustered data:")
print(data[[
    "PRODUCT DESCRIPTION",
    "HS CODE", "Chapter Desc",
    "SUB CATEGORY"
]].sample(10, random_state=42))

# --- Cluster size counts ---
print("\nCluster distribution:")
print(data["SUB CATEGORY"].value_counts())

# --- Stats by cluster ---
cluster_stats = data.groupby("SUB CATEGORY").agg({
    "HS CODE": "nunique",          # how many unique HS codes
    "PRODUCT DESCRIPTION": "count",# total rows
    "FOB INR": ["mean", "sum"], # avg and total FOB
    "QUANTITY": "mean"             # avg quantity (raw, unnormalised)
}).reset_index()

# Rename columns nicely
cluster_stats.columns = [
    "SUB CATEGORY",
    "Unique HS Codes",
    "Total Shipments",
    "Avg FOB (INR)",
    "Total FOB (INR)",
    "Avg Quantity"
]

print("\n📊 Cluster-level stats:")
print(cluster_stats.head(10))


In [ ]:
cluster_stats

In [ ]:
data.info()

In [ ]:
data2=data[['SB DATE', 'FOB INR', 'QUANTITY', 'QUANTITY UNIT' , 'IMPORTER', 'EXPORTER', 'HS CODE','Sub Heading',"Heading Desc","Sub Heading Desc","Tariff Desc","CATEGORY",'SUB CATEGORY', 'PRODUCT DESCRIPTION','Usage','Processing','Material','Weave/Construction','Other', 'FOREIGN COUNTRY', 'FOREIGN PORT', 'INDIAN PORT', 'CHAPTER']]

In [ ]:
data2['QUANTITY UNIT'].unique()

In [ ]:
data2.to_excel("clean_52081290.xlsx",index=False)

# code for Units

In [ ]:
# --- Step A: Normalisation mapping ---
unit_map = {
    "PCS": ("PIECES", 1),
    "NOS": ("PIECES", 1),
    "PRS": ("PIECES", 1),    # pairs, can adjust to 2 pcs if needed
    "DOZ": ("PIECES", 12),
    "SET": ("PIECES", 1),    # assumption
    "PAC": ("PIECES", 1),    # packet
    "BOX": ("PIECES", 1),
    "CTN": ("PIECES", 1),    # carton
    "LOT": ("PIECES", 1),    # ambiguous, leave as pieces for now
    "GRS": ("PIECES", 144),
    "THD": ("PIECES", 1000),

    "MTR": ("METRES", 1),
    "FTS": ("METRES", 0.3048), # feet → metres
    "KME": ("METRES", 1000),   # kilometre → metres
    "CMS": ("METRES", 0.01),   # centimetre → metres
    "ROL": ("METRES", 1),      # rolls need avg conversion if available
    "YDS": ("METRES", 0.9144),         # yards → metres

    "CBM": ("CUBIC_METRES", 1),        # cubic metres
    "CTM": ("CUBIC_METRES", 1.133),    # cubic tons (≈1.133 m³) – approx

    "KGS": ("KILOGRAMS", 1),
    "KGA": ("KILOGRAMS", 1),
    "MTS": ("KILOGRAMS", 1000),
    "TON": ("KILOGRAMS", 1000),
    "LBS": ("KILOGRAMS", 0.453592),
    "GMS": ("KILOGRAMS", 0.001),
    "QTL": ("KILOGRAMS", 100),

    "SQM": ("SQUARE_METRES", 1),
    "SQF": ("SQUARE_METRES", 0.092903), # sq ft → sq m

    "BAG": ("BAG", 1),     # ambiguous
    "DRM": ("DRUM", 1),    # ambiguous
    "THD": ("THREAD", 1),  # unclear
    "GRS": ("GROSS", 144)  # 1 gross = 12 dozen = 144 pcs
}

def normalise_quantity(row):
    unit = str(row["QUANTITY UNIT"]).upper().strip()
    qty = row["QUANTITY"]
    if unit in unit_map:
        new_unit, factor = unit_map[unit]
        return qty * factor, new_unit
    else:
        return qty, unit  # keep as-is if not in mapping

# Apply transformation
data[["NORMALISED_QTY", "NORMALISED_UNIT"]] = data.apply(
    lambda row: pd.Series(normalise_quantity(row)), axis=1
)

print(data[["QUANTITY UNIT", "QUANTITY", "NORMALISED_QTY", "NORMALISED_UNIT"]].sample(10))
print("Unique normalised units:", data["NORMALISED_UNIT"].unique())


      QUANTITY UNIT   QUANTITY  NORMALISED_QTY NORMALISED_UNIT
68875           MTR    2573.00       2573.0000          METRES
49861           MTR     419.00        419.0000          METRES
29885           YDS     557.00        509.3208          METRES
20921           YDS    2352.00       2150.6688          METRES
64638          SQM     6259.92       6259.9200   SQUARE_METRES
68337           SQM   89188.20      89188.2000   SQUARE_METRES
45996           SQM   56992.32      56992.3200   SQUARE_METRES
43321           SQM  128382.00     128382.0000   SQUARE_METRES
33178           MTR     600.00        600.0000          METRES
69373           NOS      60.00         60.0000          PIECES
Unique normalised units: ['METRES' 'PIECES' 'SQUARE_METRES' 'KILOGRAMS' 'BAG' 'GROSS'
 'CUBIC_METRES']


/tmp/ipython-input-3983529135.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[["NORMALISED_QTY", "NORMALISED_UNIT"]] = data.apply(
/tmp/ipython-input-3983529135.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[["NORMALISED_QTY", "NORMALISED_UNIT"]] = data.apply(


In [ ]:
# --- Step C: AVG FOB PER UNIT ---
data["AVG FOB PER UNIT"] = data["FOB INR"] / data["NORMALISED_QTY"]

# --- Step D: YEAR from SHIPPING_BILL_DATE ---
data["YEAR"] = pd.to_datetime(data["SB DATE"]).dt.year

# --- Step E: FOB SHORT (in crores with 1 decimal) ---
data["FOB SHORT"] = data["FOB INR"] / 1e7
data["FOB SHORT"] = data["FOB SHORT"].round(1)  # 1 decimal

/tmp/ipython-input-1430353785.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["AVG FOB PER UNIT"] = data["FOB INR"] / data["NORMALISED_QTY"]


In [ ]:
data.to_excel("clean_52081290_final.xlsx",index=False)

In [ ]:
# Chapter-aware pallet mapping
chapter_pallet_capacity = {
    # Chapter 1–5: Live animals, meat, fish, dairy, etc.
    (range(1, 6), "KILOGRAMS"): 1000,   # frozen cartons, ~1T per pallet
    (range(1, 6), "PIECES"): 500,       # eggs, small packs

    # Chapter 6–14: Agriculture, plants, raw materials
    (range(6, 15), "KILOGRAMS"): 900,   # sacks/bags of grains
    (range(6, 15), "BAGS"): 20,         # 20 x 50kg bags

    # Chapter 15–24: Oils, processed food, beverages
    (range(15, 25), "LITRES"): 1000,    # 200L drums (5 per pallet)
    (range(15, 25), "DRUMS"): 4,

    # Chapter 25–27: Minerals, fuels
    (range(25, 28), "KILOGRAMS"): 1200, # denser bags/cartons

    # Chapter 28–38: Chemicals, pharma
    (range(28, 39), "PIECES"): 12000,   # pharma blister packs
    (range(28, 39), "VIALS"): 8000,
    (range(28, 39), "TUBES"): 20000,
    (range(28, 39), "KILOGRAMS"): 1000, # powder bags
    (range(28, 39), "DRUMS"): 4,

    # Chapter 39–40: Plastics & rubber
    (range(39, 41), "KILOGRAMS"): 800,
    (range(39, 41), "CUBIC_METRES"): 1.2,

    # Chapter 41–43: Leather, hides, furs
    (range(41, 44), "SQUARE_METRES"): 1200,

    # Chapter 44–45: Wood, cork
    (range(44, 46), "SQUARE_METRES"): 1500,
    (range(44, 46), "CUBIC_METRES"): 1.2,
}
def get_pallet_capacity(chapter, unit):
    for (chap_range, u), cap in chapter_pallet_capacity.items():
        if chapter in chap_range and unit == u:
            return cap
    return None

def estimate_pallets(row):
    norm_qty, norm_unit = row["NORMALISED_QTY"], row["NORMALISED_UNIT"]
    chapter = int(str(row["HS CODE"]).zfill(8)[:2])  # first 2 digits = chapter
    cap = get_pallet_capacity(chapter, norm_unit)

    if cap:
        pallets = norm_qty / cap
        containers = pallets / 20  # 40ft container ~20 pallets
        return pallets, containers
    else:
        return None, None

data[["PALLETS", "CONTAINERS_40FT"]] = data.apply(
    lambda row: pd.Series(estimate_pallets(row)), axis=1
)

print(data[["PALLETS", "CONTAINERS_40FT", "NORMALISED_QTY", "NORMALISED_UNIT"]].sample(10))

In [ ]:
# --- Ensure FOB is numeric ---
data2["FOB INR"] = pd.to_numeric(data2["FOB INR"], errors="coerce")

# --- Group by HS code (6-digit) + Target Market (Importer Country) ---
shipment_stats = (
    data2.groupby(["Sub Heading", "FOREIGN COUNTRY", "NORMALISED_UNIT"])
    .agg(
        Avg_FOB_per_Shipment=("FOB INR", "mean"),  # average FOB per shipment
        Avg_FOB_per_Unit=("FOB INR", lambda x: (x.sum() / data2.loc[x.index, "NORMALISED_QTY"].sum())
                          if data2.loc[x.index, "NORMALISED_QTY"].sum() > 0 else None),
        Avg_Units_per_Shipment=("NORMALISED_QTY", "mean"),
        Total_FOB=("FOB INR", "sum"),
        Total_Units=("NORMALISED_QTY", "sum"),
        Total_Shipments=("FOB INR", "count")  # now using count as proxy for number of shipments
    )
    .reset_index()
)

# --- Round for readability ---
shipment_stats["Avg_FOB_per_Shipment"] = shipment_stats["Avg_FOB_per_Shipment"].round(2)
shipment_stats["Avg_FOB_per_Unit"] = shipment_stats["Avg_FOB_per_Unit"].round(2)
shipment_stats["Avg_Units_per_Shipment"] = shipment_stats["Avg_Units_per_Shipment"].round(2)
shipment_stats["Total_FOB"] = shipment_stats["Total_FOB"].round(2)

shipment_stats.head()


In [ ]:
print("📊 Typical Shipment Size Stats (first 10 rows):")
print(shipment_stats.head(10))

# Save to Excel for inspection
shipment_stats.to_excel("typical_shipment_size.xlsx", index=False)


In [ ]:
# --- Ensure FOB is numeric ---
data["FOB INR"] = pd.to_numeric(data["FOB INR"], errors="coerce")

# --- Group by CATEGORY + SUB CATEGORY + NORMALISED_UNIT ---
shipment_stats = (
    data.groupby(["CATEGORY", "NORMALISED_UNIT"])
    .agg(
        Total_Shipments=("SB NO", "nunique"),   # distinct shipping bills
        Avg_FOB_per_Shipment=("FOB INR", "mean"),
        Avg_FOB_per_Unit=("FOB INR", lambda x: (
            x.sum() / data.loc[x.index, "NORMALISED_QTY"].sum()
        ) if data.loc[x.index, "NORMALISED_QTY"].sum() > 0 else None),
        Avg_Units_per_Shipment=("NORMALISED_QTY", "mean"),
        Total_FOB=("FOB INR", "sum"),
        Total_Units=("NORMALISED_QTY", "sum")
    )
    .reset_index()
)

# --- Round for readability ---
shipment_stats["Avg_FOB_per_Shipment"] = shipment_stats["Avg_FOB_per_Shipment"].round(2)
shipment_stats["Avg_FOB_per_Unit"] = shipment_stats["Avg_FOB_per_Unit"].round(2)
shipment_stats["Avg_Units_per_Shipment"] = shipment_stats["Avg_Units_per_Shipment"].round(2)

# --- Quick preview ---
print(shipment_stats.head(10))


In [ ]:
shipment_stats.to_excel("typical_shipment_size.xlsx", index=False)

In [ ]:

# --- Step 1: Aggregate by IMPORTER/EXPORTER ---
agg = data.groupby(["IMPORTER", "EXPORTER"]).agg(
    total_value=("FOB INR", "sum"),
    total_quantity=("NORMALISED_QTY", "sum"),
    num_shipments=("FOB INR", "count"),
    avg_value=("FOB INR", "mean"),
    first_date=("SB DATE", "min"),
    last_date=("SB DATE", "max"),
).reset_index()

# --- Step 2: Calculate shipment frequency (per month) ---
agg["months_active"] = ((agg["last_date"] - agg["first_date"]).dt.days / 30).clip(lower=1)
agg["shipments_per_month"] = agg["num_shipments"] / agg["months_active"]

# --- Step 3: Classify market segment ---
def segment(row):
    if row["avg_value"] > 10_00_000 and row["shipments_per_month"] < 2:
        return "Bulk B2B (Large, infrequent)"
    elif row["avg_value"] < 2_00_000 and row["shipments_per_month"] > 3:
        return "B2C / SMEs (Small, frequent)"
    else:
        return "Mixed / Mid-size"

agg["Market Segment"] = agg.apply(segment, axis=1)

# --- Final segmented dataset ---
print(agg[["IMPORTER", "EXPORTER", "Market Segment", "avg_value", "shipments_per_month"]].head())


In [ ]:
agg[["IMPORTER", "EXPORTER", "Market Segment", "avg_value", "shipments_per_month"]].to_excel(
    "market_segments_review.xlsx",
    index=False
)
